In [1]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score

In [2]:
rf = joblib.load(
    "random_forest_model.pkl"
)
feature_columns = joblib.load("feature_columns.pkl")

In [3]:
score_data = pd.read_csv("D:\Projects\Project-Cab_cancellation_prediction\YourCabs_score.csv\YourCabs_score.csv")

In [4]:
score_data.info()
score_data.head()
score_data.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   10000 non-null  int64  
 1   user_id              10000 non-null  int64  
 2   vehicle_model_id     10000 non-null  int64  
 3   package_id           1785 non-null   float64
 4   travel_type_id       10000 non-null  int64  
 5   from_area_id         9980 non-null   float64
 6   to_area_id           7850 non-null   float64
 7   from_city_id         3712 non-null   float64
 8   to_city_id           365 non-null    float64
 9   from_date            10000 non-null  object 
 10  to_date              5873 non-null   object 
 11  online_booking       10000 non-null  int64  
 12  mobile_site_booking  10000 non-null  int64  
 13  booking_created      10000 non-null  object 
 14  from_lat             9980 non-null   float64
 15  from_long            9980 non-null   

id                         0
user_id                    0
vehicle_model_id           0
package_id              8215
travel_type_id             0
from_area_id              20
to_area_id              2150
from_city_id            6288
to_city_id              9635
from_date                  0
to_date                 4127
online_booking             0
mobile_site_booking        0
booking_created            0
from_lat                  20
from_long                 20
to_lat                  2150
to_long                 2150
Unnamed: 18            10000
Unnamed: 19            10000
dtype: int64

In [5]:
score_ids = score_data['id'].copy()

In [6]:
score_data.drop(
    ['id', 'from_city_id', 'to_city_id'],
    axis=1,
    inplace=True
)

In [7]:
score_data['package_id'] = score_data['package_id'].fillna(0)

score_data['from_area_id'] = score_data['from_area_id'].fillna(
    score_data['from_area_id'].mode()[0]
)
score_data['to_area_id'] = score_data['to_area_id'].fillna(0)

In [8]:
score_data['from_date'] = pd.to_datetime(score_data['from_date'])

score_data['booking_created'] = pd.to_datetime(score_data['booking_created'])

score_data['to_date'] = pd.to_datetime(
    score_data['to_date'],
    errors='coerce'
)

In [9]:
booking_count = score_data.groupby('user_id').size()

score_data['customer_booking_count'] = score_data['user_id'].map(
    booking_count
)

In [10]:
score_data['ride_hour'] = score_data['from_date'].dt.hour

score_data['ride_dayofweek'] = score_data['from_date'].dt.dayofweek

score_data['booking_hour'] = score_data['booking_created'].dt.hour

score_data['booking_dayofweek'] = score_data['booking_created'].dt.dayofweek

score_data['booking_advance_hours'] = (
    score_data['from_date']
    - score_data['booking_created']
).dt.total_seconds() / 3600

score_data['is_same_day_booking'] = (
    score_data['from_date'].dt.date
    ==
    score_data['booking_created'].dt.date
).astype(int)

In [11]:
def categorize_booking(hours):
    if hours <= 1:
        return 'Urgent'
    elif hours <= 5:
        return 'SameDay'
    elif hours <= 24:
        return 'Regular'
    else:
        return 'Advance'

score_data['booking_nature'] = (
    score_data['booking_advance_hours']
    .apply(categorize_booking)
)

In [12]:
from sklearn.preprocessing import LabelEncoder

le_booking = LabelEncoder()

score_data['booking_nature'] = (
    le_booking.fit_transform(score_data['booking_nature'])
)

In [13]:
score_data['is_VMID_12'] = (
    score_data['vehicle_model_id'] == 12
).astype(int)

score_data.drop(
    'vehicle_model_id',
    axis=1,
    inplace=True
)

In [14]:
score_data.drop(
    ['user_id', 'from_date', 'to_date', 'booking_created'],
    axis=1,
    inplace=True
)

In [15]:
print(feature_columns == score_data.columns.tolist())
print(feature_columns)
print(score_data.columns.tolist())

False
['package_id', 'travel_type_id', 'from_area_id', 'to_area_id', 'online_booking', 'mobile_site_booking', 'customer_booking_count', 'ride_hour', 'ride_dayofweek', 'booking_hour', 'booking_dayofweek', 'booking_advance_hours', 'is_same_day_booking', 'booking_nature', 'is_VMID_12']
['package_id', 'travel_type_id', 'from_area_id', 'to_area_id', 'online_booking', 'mobile_site_booking', 'from_lat', 'from_long', 'to_lat', 'to_long', 'Unnamed: 18', 'Unnamed: 19', 'customer_booking_count', 'ride_hour', 'ride_dayofweek', 'booking_hour', 'booking_dayofweek', 'booking_advance_hours', 'is_same_day_booking', 'booking_nature', 'is_VMID_12']


In [16]:
score_data.drop(
    ['from_lat', 'from_long',
       'to_lat', 'to_long', 'Unnamed: 18', 'Unnamed: 19'],
    axis=1,
    inplace=True
)

In [17]:
predictions = rf.predict(score_data)

probabilities = rf.predict_proba(score_data)[:, 1]

In [18]:
submission = pd.DataFrame({
    'id': score_ids,
    'Predicted_Cancellation': predictions,
    'Cancellation_Probability': probabilities
})

submission.head()

,id,Predicted_Cancellation,Cancellation_Probability
0,132516,0,0.100
1,132529,0,0.015
2,132532,0,0.125
3,132547,0,0.490
4,132548,0,0.075


In [19]:
submission.to_csv(
    'cab_cancellation_predictions.csv',
    index=False
)